# Quality Controls for the OCR output

This notebook contains test that check the completeness and the quality of the OCR'ed output before we continue processing it.

In [3]:
import os
import json
import re
from pathlib import Path
from pypdf import PdfReader
from statistics import mean, median

In [4]:
pdf_path = Path(f"../pdf_corpus/just_ottoman_pdfs/vasif_1789-1794.pdf")
output_dir = Path("gemini_output")

## Were all pages processed?

In [2]:
# Path setup
pdf_path = Path(f"../pdf_corpus/just_ottoman_pdfs/vasif_1789-1794.pdf")
output_dir = Path("gemini_output")

# Check if PDF exists
if not pdf_path.exists():
    print(f"PDF not found: {pdf_path}")
else:
    # Get number of pages in PDF
    reader = PdfReader(pdf_path)
    pdf_page_count = len(reader.pages)
    print(f"PDF has {pdf_page_count} pages")
    
    # Count JSON files in output directory
    if not output_dir.exists():
        print(f"Output directory not found: {output_dir}")
        json_file_count = 0
    else:
        json_files = list(output_dir.glob("page_*.json"))
        json_file_count = len(json_files)
        print(f"Found {json_file_count} JSON files in output directory")

        # Extract page numbers from existing JSON files
        existing_pages = set()
        for json_file in json_files:
            page_num = int(json_file.stem.split('_')[1])
            existing_pages.add(page_num)
        
        # Find the min and max page numbers from your JSON files
        if existing_pages:
            min_page = min(existing_pages)
            max_page = max(existing_pages)
        
            # Generate the expected range based on your actual page numbering
            all_pages = set(range(min_page, max_page + 1))
            missing_pages = sorted(all_pages - existing_pages)
        
        # Compare
        if pdf_page_count == json_file_count:
            print(f"All pages processed! ({pdf_page_count} pages)")
        elif pdf_page_count < json_file_count:
            print(f"More JSON files than PDF pages!")
        else:
            missing = pdf_page_count - json_file_count
            print(f"Missing {missing} pages (PDF: {pdf_page_count}, JSON files: {json_file_count})")
            if missing_pages:
                print(f"Missing pages: {missing_pages}")


PDF has 387 pages
Found 387 JSON files in output directory
All pages processed! (387 pages)


- PDF has 387 pages
- Found 387 JSON files in output directory
- All pages processed! (387 pages)

Basically this is confirming that we have one JSON for each PDF page. If we had more JSON files than PDF pages or if some PDF pages did not have JSON files, this could would flag it.


## Do all pages have a text output?

In [8]:
# Check for files with null raw_response
output_dir = Path("gemini_output")
null_responses = []

json_files = sorted(output_dir.glob("page_*.json"))
print(f"Checking {len(json_files)} files…")

for json_file in json_files:
    # read once as text and look for the null marker
    txt = json_file.read_text(encoding="utf-8")
    if '"raw_response": null' in txt or '"raw_response":null' in txt:
        # parse only when we actually need the page number
        data = json.loads(txt)
        null_responses.append({
            "file": json_file.name,
            "page": data.get("page", "unknown"),
            "timestamp": data.get("timestamp", "N/A"),
        })

total_files = len(json_files)
if not null_responses:
    print(f"All {total_files} files have text output")
else:
    print(f"Found {len(null_responses)} files with null raw_response out of {total_files} total:")
    for item in null_responses:
        print(f"  - {item['file']} (page {item['page']})")

checking 387 files…
All 387 files have text output


All 387 files have text output -> This is perfect. It means that all of our 387 pages have a text output.

If they didn't have text output, I would need to re-run the OCR code on these files. 

I was lucky with `vasif_1789-1794` and didn't need to re-run anything. But, when I was OCR'ing and testing `vasif_1794-1805` I realized that there were several files with no response. I updated my OCR loop to account for this issue and re-run if there is no response.

Now that we checked for *completeness*, we can run some further tests to get a sense of the *quality* of the output.

## Are all raw outputs JSON and do they follow the expected schema?

In [ ]:
def basic_validation(json_files):
    """Check that files have valid JSON, required top-level fields, and event structure."""
    not_json = []
    missing_page = []
    invalid_events_list = []
    files_with_issues = []

    for jf in json_files:
        with open(jf, "r", encoding="utf-8") as f:
            wrapper = json.load(f)

        raw = wrapper.get("raw_response")
        if raw is None:
            continue  # skipped in earlier check

        # Parse JSON
        cleaned = raw.strip()
        if cleaned.startswith("```"):
            lines = cleaned.splitlines()
            if lines and lines[0].startswith("```"):
                lines = lines[1:]
            if lines and lines[-1].strip().startswith("```"):
                lines = lines[:-1]
            cleaned = "\n".join(lines).strip()

        try:
            parsed = json.loads(cleaned)
        except json.JSONDecodeError:
            not_json.append(jf.name)
            continue

        # Check required top-level fields
        if "page" not in parsed:
            missing_page.append(jf.name)

        events = parsed.get("events")
        if not isinstance(events, list):
            invalid_events_list.append(jf.name)
            continue

        # Check structure of each event
        # we expect each event to be a dict with "subheading" and "body" keys
        # while either key can be empty, they must exist
        has_issues = False
        for idx, ev in enumerate(events, start=1):
            if not isinstance(ev, dict):
                has_issues = True
                break
            if "subheading" not in ev or "body" not in ev:
                has_issues = True
                break

        if has_issues:
            files_with_issues.append(jf.name)

    return {
        "not_json": not_json,
        "missing_page": missing_page,
        "invalid_events_list": invalid_events_list,
        "files_with_issues": files_with_issues,
    }

In [9]:
json_files = sorted(output_dir.glob("page_*.json"))
results = basic_validation(json_files)

print(f"Checked {len(json_files)} files\n")

if results["not_json"]:
    print(f"❌ Non-JSON: {results['not_json']}")
if results["missing_page"]:
    print(f"❌ Missing 'page' field: {results['missing_page']}")
if results["invalid_events_list"]:
    print(f"❌ Invalid 'events' list: {results['invalid_events_list']}")
if results["files_with_issues"]:
    print(f"⚠️  Files with structural issues ({len(results['files_with_issues'])}): {results['files_with_issues'][:5]}")
else:
    print("✅ All files passed basic validation")

Checked 387 files

❌ Non-JSON: ['page_154.json', 'page_192.json', 'page_216.json']
⚠️  Files with structural issues (3): ['page_399.json', 'page_409.json', 'page_511.json']


❌ Non-JSON: ['page_154.json', 'page_192.json', 'page_216.json']

Our basic validation test flagged the 3 non-JSON pages. These are the pages that we created as few-shot examples, so I am not investigating them any further. If they were files generated by Gemini, I would look into them personally.

⚠️  Files with structural issues (3): ['page_399.json', 'page_409.json', 'page_511.json']

Our basic validation checks to make sure that for each event in the events list, there are two keys: subheading and body. The values associated with these keys can be empty but the keys themselves must exist. 

In the next step, we will investigate these 3 files further.

In [10]:
def detailed_inspection(json_files):
    """
    For files flagged in basic validation, show exactly which event is missing which key.
    Pass a list of problem filenames.
    """
    output_dir = Path("gemini_output")

    for filename in json_files:
        print(f"\n{'='*60}")
        print(f"Inspecting: {filename}")
        print('='*60)

        with open(output_dir / filename, "r", encoding="utf-8") as f:
            wrapper = json.load(f)

        raw = wrapper.get("raw_response")
        if raw is None:
            print("  ⚠️  raw_response is None")
            continue

        cleaned = raw.strip()
        if cleaned.startswith("```"):
            lines = cleaned.splitlines()
            if lines and lines[0].startswith("```"):
                lines = lines[1:]
            if lines and lines[-1].strip().startswith("```"):
                lines = lines[:-1]
            cleaned = "\n".join(lines).strip()

        try:
            parsed = json.loads(cleaned)
        except json.JSONDecodeError as e:
            print(f"  ❌ JSON parse error: {e}")
            continue

        events = parsed.get("events", [])
        print(f"  Total events: {len(events)}\n")

        for idx, ev in enumerate(events, start=1):
            has_subheading = "subheading" in ev
            has_body = "body" in ev
            subheading_val = ev.get("subheading", "KEY_MISSING")
            body_preview = str(ev.get("body", "KEY_MISSING"))[:50]

            status = "✅" if (has_subheading and has_body) else "❌"
            print(f"  Event {idx}: {status}")
            print(f"    'subheading' key: {has_subheading} → {subheading_val!r}")
            print(f"    'body' key:       {has_body} → {body_preview}...")
            if not has_subheading or not has_body:
                print(f"    All keys: {list(ev.keys())}")
            print()

In [11]:
# If there are issues, inspect them
if results["files_with_issues"]:
    detailed_inspection(results["files_with_issues"])
else:
    print("\n✅ All checks passed!")


Inspecting: page_399.json
  Total events: 3

  Event 1: ✅
    'subheading' key: True → None
    'body' key:       True → mu'âmele ve ancak Bender Kalası binâsını havâle bu...

  Event 2: ❌
    'subheading' key: True → "Rumeli ve Anadolu'da vâki‘ memâlik-i Pâdişâhî'de 'adl ü dâdın intişârı ve fukarâ vü zuʻafânın ıslâh-[152a]kârı ve sâye-i şefekat vâyelerinde asayiş ü râhatla istikrârı matlûb bulup, zuhûr-ı mezâlim ise fî-ekseri'l-evkāt hükkâm u a'yân nâmıyla memleketlere müstevlî olan li’âm-ı enâmdan neş'et eyleyüp, şöyle ki, bir senelik mesârif nâmıyla ehâlî-yi kazâya tevzî‘ eyledikleri mebâliğin ‘öşrüyle umûr-ı kazâ idâre olunup, fazlası miyânelerinde münkasim ve bu sebeble reʻâyâda tâb u tüvân kalmayup, tekâlîf-i şer‘iyyeyi bile edâda tâkatleri mün‘adim olup, giderek bu zulmün istimrârı vîrân-ı memâlike sebeb ve belki mûceb-i şûr u şegab olacağı rûz u şeb hâst-gâr-ı rızâ-yı Rabb olan Şehinşâh-ı vâlâ-neseb hazretlerinin meczûmu olmağla, fî-mâ-ba'd altı mâhda bir kerre her kazânın sic

```python
============================================================
Inspecting: page_399.json
============================================================
  Total events: 3

  Event 1: ✅
    'subheading' key: True → None
    'body' key:       True → mu'âmele ve ancak Bender Kalası binâsını havâle bu...

  Event 2: ❌
    'subheading' key: True → "Rumeli ve Anadolu'da vâki‘ memâlik-i Pâdişâhî'de 'adl ü dâdın intişârı ve fukarâ vü zuʻafânın ıslâh-[152a]kârı ve sâye-i şefekat vâyelerinde asayiş ü râhatla istikrârı matlûb bulup, zuhûr-ı mezâlim ise fî-ekseri'l-evkāt hükkâm u a'yân nâmıyla memleketlere müstevlî olan li’âm-ı enâmdan neş'et eyleyüp, şöyle ki, bir senelik mesârif nâmıyla ehâlî-yi kazâya tevzî‘ eyledikleri mebâliğin ‘öşrüyle umûr-ı kazâ idâre olunup, fazlası miyânelerinde münkasim ve bu sebeble reʻâyâda tâb u tüvân kalmayup, tekâlîf-i şer‘iyyeyi bile edâda tâkatleri mün‘adim olup, giderek bu zulmün istimrârı vîrân-ı memâlike sebeb ve belki mûceb-i şûr u şegab olacağı rûz u şeb hâst-gâr-ı rızâ-yı Rabb olan Şehinşâh-ı vâlâ-neseb hazretlerinin meczûmu olmağla, fî-mâ-ba'd altı mâhda bir kerre her kazânın sicillât-ı mesafiri Âsitâne-i sa‘âdet'e getirdilüp, me'mûrlar huzûrunda ta'dîl ve sıhhate makrûn olan mesârif isbât ve mevzû'ât-ı hükkâmdan olan emvâl-i zâyide hatt u tenzîl olunmak zımnında, Memâlik-i mahrûse'ye neşr-i evâmir ve ta‘yîn-i mübâşir kılınup, bu vesîle-i cemîle ile eyâdî-yi zulme kûtâh ve me’lûf-ı ekl ü bel‘ olan rû-siyâhlar mübtelâ-yı nâliş-i cângâh oldular. Zikrolunan defâtirin tenkīhi ve ıskāt-ı zevâyid ve hazf-ı avâyid ile tashîhi ahvâl-i memâlike muttali‘ bir zâta tahsîs olunmak lâzım geldiğine binâ'en, Anadolu Defterleri, Şıkk-ı Evvel-i sâbık Mehmed Şerîf Efendi'ye ve Rumeli Defterleri, Yenişehirli Mustafa Bey'e tefvîz olundu. Nazm: Geldi mîzân cemâl-i âraya ey Yusuf Hasan! Korkarım çok keşenk ipliği bâzâra çıkar."
    'body' key:       False → KEY_MISSING...
    All keys: ['subheading']

  Event 3: ✅
    'subheading' key: True → None
    'body' key:       True → Sultân Bâyezid'de vâki' 'Abdulkerim Beyzâde'nin hâ...


============================================================
Inspecting: page_409.json
============================================================
  Total events: 4

  Event 1: ✅
    'subheading' key: True → None
    'body' key:       True → i mücellâ-yı bürûz olup, yevm-i mezkûrda mükerrere...

  Event 2: ✅
    'subheading' key: True → 'İhrâc-ı mevâcib ve vukūʻât-ı sâyire'
    'body' key:       True → Şehr-i mezkûr hılâlinde tavâyif-i ‘askeriyyenin kı...

  Event 3: ❌
    'subheading' key: True → "Humbaracı ve Lağımcı ocağlarına müteʻayyinân-ı Devlet-i ‘aliyye'den birer ağa nasb ve mikdâr-ı vâfî meʻâş ile ‘ârıza-i zarûretleri selb olunmak ocağlarına hürmet ve bekā-yı nizâmlarına ‘illet olacağı zâhir olmağla Dergâh-ı ‘âlî kapucu-başılarından sabıkā Silahdâr Ağası Kethudâ [158b] Ahmed Ağa senevî on beş kîse meʻâş ile Humbaracı Ocağı'na ve Dergâh-ı ‘âlî gediklülerinden Mehmed Emîn Ağa senevî on kîse meʻâş ile Lağımcı Ocağı'na Ağa nasb olunup, şehr-i mezkûrun yigirmi üçüncü günü ilbâs-ı hil'at ile taltîf ve kānûn-nâme mûcebince idare-i umûr etmeleri tenbîhâtıyla sâmi‘a-i izʻânları teşnîf olundu."
    'body' key:       False → KEY_MISSING...
    All keys: ['subheading']

  Event 4: ✅
    'subheading' key: True → 'Tecdîd-i pûşîde-i Hazret-i Mevlânâ ve iksâ-yı merâkıd-ı yârân-ı ô'
    'body' key:       True → Mahrûse-i Konya'da gunûde-i pister-i zarîh ve dâr-...


============================================================
Inspecting: page_511.json
============================================================
  Total events: 2

  Event 1: ✅
    'subheading' key: True → None
    'body' key:       True → Binâ-berîn sâbıkā Cebeciler Kâtibi Eginli Mustafâ ...

  Event 2: ❌
    'subheading' key: True → "Sâbıkā Sipâh Kâtibi Râşid Efendi ikāle-i ‘asrat ve izâle-i gubâr-ı nefret ümniyyesiyle bu hıdmet-i şeref-bahşın rü'yetine taleb-kâr ve şiddet-i şevk ü hâhişi hüsn-i terbiye ile hâk-pâ-yi hazret-i Şehriyârî'ye 'arz u işʻâr olundukda, [218a] Emânet-i Binâ el-yevm İsmaʻîl taraflarında beste-miyân-ı hıdmet olan Mu'ammer Ağa'ya ihâle olunup, Efendi-yi mümâ ileyhin nezâret-i mücerrede ile iktifâsı vehle-i ûlâda irâde olunmuşiken, müteveffâ Nuʻmân Bey'in me'mûriyyet-i mürettebesi üzere taʻyîn olunmak ilticâsında olduğu tekrâr 'arz-ı Der-bâr-ı ‘atûfet-medâr kılınup, ol vechile me'mûriyyeti muvâfık-ı re'y-i Sâmî olmağla, der-‘akab da‘vet ve Baş-muhasebe Pâyesi'yle ilbâs-ı hilʻat ve birkaç gün zarfında tedârükünü görüp, izhâr-ı memnuniyyet eyleyerek İsmaʻîl cânibine ‘azîmet eyledi. Kalʻa-i mezkûrenin ‘amele"
    'body' key:       False → KEY_MISSING...
    All keys: ['subheading']

```
---

This output very clearly shows that in these 3 files, some of the events were marked as subheadings. Let's look at the pages themselves

![img](page_409.png)

As you can see in this one, the model thought that the "Humbaracı ve Lağımcı" written in bold was a new subheading but it could not determine where it ended and the body of that event started. Hence, it saved the whole text as a subheading until it hit the next real subheading. 

Both of the other pages have similar issues where text that is meant to be just continuation of the previous event was deemed to be a subheading.

Since we know what the issue is, we can either fix this manually or we can fix this using a function. However, that is best reserved for a later stage of this pipeline. For now we will flag these events and return to them in post-processing.

## Are all events properly structured?

In the way that we coded the extraction, if the first event does not have a subheading, it is ok because it is a continuation from the previous page. However, any subsequent event must have a subheading. 

In [14]:
def check_consecutive_nulls(json_files):
    """Flag files where two events in a row have null subheadings."""
    files_with_consecutive_nulls = []
    
    for jf in json_files:
        with open(jf, "r", encoding="utf-8") as f:
            wrapper = json.load(f)
        
        raw = wrapper.get("raw_response")
        if raw is None:
            continue
        
        # Strip fences, parse
        cleaned = raw.strip()
        if cleaned.startswith("```"):
            lines = cleaned.splitlines()
            if lines and lines[0].startswith("```"):
                lines = lines[1:]
            if lines and lines[-1].strip().startswith("```"):
                lines = lines[:-1]
            cleaned = "\n".join(lines).strip()
        
        try:
            parsed = json.loads(cleaned)
        except json.JSONDecodeError:
            continue
        
        events = parsed.get("events", [])
        
        # Check for consecutive events with null subheadings
        for idx in range(len(events) - 1):
            curr = events[idx]
            next_ev = events[idx + 1]
            
            if (isinstance(curr, dict) and isinstance(next_ev, dict) and
                curr.get("subheading") is None and next_ev.get("subheading") is None):
                files_with_consecutive_nulls.append({
                    "file": jf.name,
                    "event_pair": (idx + 1, idx + 2)  # 1-indexed for readability
                })
    
    return files_with_consecutive_nulls

In [15]:
consecutive_null_issues = check_consecutive_nulls(json_files)
if consecutive_null_issues:
    print(f"\n⚠️ Files with consecutive events lacking subheadings:")
    for item in consecutive_null_issues:
        print(f"  - {item['file']} (events {item['event_pair'][0]} and {item['event_pair'][1]})")
else:
    print("\n✅ No files with consecutive null subheadings")


⚠️ Files with consecutive events lacking subheadings:
  - page_177.json (events 1 and 2)
  - page_220.json (events 1 and 2)
  - page_227.json (events 1 and 2)
  - page_265.json (events 1 and 2)
  - page_269.json (events 1 and 2)
  - page_299.json (events 1 and 2)
  - page_303.json (events 1 and 2)
  - page_378.json (events 1 and 2)
  - page_378.json (events 2 and 3)
  - page_395.json (events 1 and 2)
  - page_403.json (events 2 and 3)
  - page_420.json (events 1 and 2)


⚠️ Files with consecutive events lacking subheadings:
  - page_177.json (events 1 and 2)
  - page_220.json (events 1 and 2)
  - page_227.json (events 1 and 2)
  - page_265.json (events 1 and 2)
  - page_269.json (events 1 and 2)
  - page_299.json (events 1 and 2)
  - page_303.json (events 1 and 2)
  - page_378.json (events 1 and 2)
  - page_378.json (events 2 and 3)
  - page_395.json (events 1 and 2)
  - page_403.json (events 2 and 3)
  - page_420.json (events 1 and 2)
  

In the cases where events 1 and 2 are flagged, it is very likely that event 1 is a continuation from the previous page. While there might be some outlier, event 1 having a null subheading is in line with our extraction schema. In this case, we will assume that event 1 is fully formed and event 2 is the problem.

Let's print them and see what is happening in more detail:

In [25]:
def detailed_consecutive_nulls_inspection(consecutive_null_issues):
    """
    For files flagged with consecutive null subheadings, show the boundary between them.
    Assumes event 1 (first flagged) is continuation and focuses on event 2 (second flagged) as the problem.
    """
    output_dir = Path("gemini_output")
    
    for item in consecutive_null_issues:
        filename = item["file"]
        event_pair = item["event_pair"]  # e.g., (1, 2) or (2, 3)
        
        print(f"\n{'='*60}")
        print(f"Inspecting: {filename} (events {event_pair[0]} and {event_pair[1]})")
        print('='*60)
        
        with open(output_dir / filename, "r", encoding="utf-8") as f:
            wrapper = json.load(f)
        
        raw = wrapper.get("raw_response")
        if raw is None:
            print("  ⚠️  raw_response is None")
            continue
        
        cleaned = raw.strip()
        if cleaned.startswith("```"):
            lines = cleaned.splitlines()
            if lines and lines[0].startswith("```"):
                lines = lines[1:]
            if lines and lines[-1].strip().startswith("```"):
                lines = lines[:-1]
            cleaned = "\n".join(lines).strip()
        
        try:
            parsed = json.loads(cleaned)
        except json.JSONDecodeError as e:
            print(f"  ❌ JSON parse error: {e}")
            continue
        
        events = parsed.get("events", [])
        
        # Get indices (convert from 1-indexed to 0-indexed)
        first_idx = event_pair[0] - 1
        second_idx = event_pair[1] - 1
        
        # Show end of first flagged event
        first_ev = events[first_idx]
        if isinstance(first_ev, dict):
            first_body = first_ev.get("body", "")
            first_end = first_body[-300:] if len(first_body) > 300 else first_body
            print(f"\n  Event {event_pair[0]} (null subheading - continuation):")
            print(f"    ...{first_end!r}\n")
        
        # Show beginning of second flagged event
        second_ev = events[second_idx]
        if isinstance(second_ev, dict):
            second_body = second_ev.get("body", "")
            second_start = second_body[:300] if len(second_body) > 300 else second_body
            print(f"  Event {event_pair[1]} (null subheading - PROBLEM):")
            print(f"    {second_start!r}...\n")

In [26]:
consecutive_null_issues = check_consecutive_nulls(json_files)
if consecutive_null_issues:
    # Now inspect them in detail
    detailed_consecutive_nulls_inspection(consecutive_null_issues)
else:
    print("\n✅ No files with consecutive null subheadings")


Inspecting: page_177.json (events 1 and 2)

  Event 1 (null subheading - continuation):
    ..."i tedbîr eylediklerine binâ’en, o hıdmete fi'l-hâl tahsîs ve takviye-i bâzû-yı iktidârına medâr olan mühimmât ü levâzımât ne ise, taleb ü iddiʻâda mûmâ ileyh terhîs olunup, ol bâbda lâzım gelanlere ‘alâ-vechi'd-dakkati tenbîh ü tavsiye ve erbâb-ı meşverete ruhsat ʻavd verilüp, meclis tahliye olundu."

  Event 2 (null subheading - PROBLEM):
    "Kethudâ-yı Sadrıaʻzamî olan Hasan Efendi'nin kuvvet-i baht ile tâliʻi muhkem u saht ve her mevsimde merâtib-i âliyyeye irtikā ile cebel-i Kāf, irâd-ı devleti fe’s-i iʻtisâf ile naht eylediğinden gayri, derece-i ʻakl ü şuʻûrdan sâkıt ve kâffe-i nâsa min gayr-i sebeb bâgız u sâhıt olup, bu makūle sıfâ"...


Inspecting: page_220.json (events 1 and 2)

  Event 1 (null subheading - continuation):
    ..." Mîr-i mîrânlık ile çerâğ eylediği ʻAli Paşa'nın terfîʻ kadrleri matlûb olunduğuna binâ'en, ikisine dahi eşref-i merâtib-i Devlet-i ʻaliyye'den olan câh-

Let's look at page 220 in more detail:

![img](page_220.png)

```python
============================================================
Inspecting: page_220.json (events 1 and 2)
============================================================

  Event 1 (null subheading - continuation):
    ..." Mîr-i mîrânlık ile çerâğ eylediği ʻAli Paşa'nın terfîʻ kadrleri matlûb olunduğuna binâ'en, ikisine dahi eşref-i merâtib-i Devlet-i ʻaliyye'den olan câh-ı vâla-yı vezâret verilüp, Köstendil Sancağı'yla kemâ-kân İsakçı muhâfazası ʻOsmân Paşa'ya tenbîh ü telkīn ve ʻAli Paşa Ada imdâdına taʻyîn olundu."

  Event 2 (null subheading - PROBLEM):
    "Ser-çukadâr-ı hazret-i Şehriyârî Rikâb-ı kâm-yâb-ı Hüsrevâne'ye rûh-sûde oldukda, Sadrıaʻzam'ın kûze-i hâfızasına îdâ eylediği mültemesâtı yek-be-yek şümârende-i beyân-ı hüsn-i taʻbîr ve fi'l-hakīka makām-ı Sadâret ve Sipah-sâlârî'de bulunan zevât-ı hazêrâtına istiklâl-i tâm verilmek rü’yet-i umûr-ı"...
```

Here we can see clearly that the issue is with the model marking the beginning of a new paragraph as if it is a new event. Best way to approach this will be to merge them together in the post-processing. Or we can just leave it be, because in the end when we move from individual PDF pages to events, those events without subheadings will be treated as if they are continuations of the previous events.

## Are all the years correctly identified?

This is relatively straighforward. We will get all the events with only subheading and filter them by a year regex. This needs to be to some extent specific to each book but it will contain some version of 'vekayi'. then we will compare this with the number of years we expect based on the books coverage.

In this case, `vasif_1789-1794` covers the Hijri years 1203-1209, meaning that the text begins during the year 1203 (so no marker for that one) and then one marker for each of the years 1204, 1205, 1206, 1207, 1208, and 1209.

Cross referencing it with the book, we know that they appear on the following pages 

1204: 192 , 1205: 242, 1206: 335, 1207: 383 , 1208: 434, and 1209: 518

In [7]:
def check_year_markers(json_files, expected_count):
    """
    Find events with only a subheading matching the VEKĀYİ pattern
    and compare the count against the expected number of year markers.
    """
    VEKAYI_PATTERN = re.compile(r"VEK[AÂĀ]Y[İI]", re.IGNORECASE)
    
    found_markers = []

    for jf in json_files:
        with open(jf, "r", encoding="utf-8") as f:
            wrapper = json.load(f)

        raw_response = wrapper.get("raw_response")
        if raw_response is None:
            continue

        cleaned = raw_response.strip()
        if cleaned.startswith("```"):
            lines = cleaned.splitlines()
            if lines and lines[0].startswith("```"):
                lines = lines[1:]
            if lines and lines[-1].strip().startswith("```"):
                lines = lines[:-1]
            cleaned = "\n".join(lines).strip()

        try:
            parsed = json.loads(cleaned)
        except json.JSONDecodeError:
            continue

        for ev in parsed.get("events", []):
            if not isinstance(ev, dict):
                continue
            sub = ev.get("subheading") or ""
            body = ev.get("body")
            body_is_empty = body is None or body == "null" or body == ""

            if body_is_empty and VEKAYI_PATTERN.search(sub):
                found_markers.append({
                    "file": jf.name,
                    "page": parsed.get("page"),
                    "subheading": sub
                })

    # ── Report ────────────────────────────────────────────────
    status = "✅" if len(found_markers) == expected_count else "❌"
    print(f"{status} Found {len(found_markers)} year marker(s), expected {expected_count}\n")

    for m in found_markers:
        print(f"  page {m['page']:>4}: {m['subheading']!r}")

    return found_markers

In [8]:
json_files = sorted(Path("gemini_output").glob("*.json"))
markers = check_year_markers(json_files, expected_count=6)

❌ Found 5 year marker(s), expected 6

  page  242: "VEKĀYİʻ-İ SENE HAMSÜ VE MİʼETEYN VE ELF MİN HİCRETİ MEN LEHU'L-ʻİZZÜ VE'Ş-ŞEREF"
  page  335: "VEKÂYİ-İ SENE SİTTE VE Mİ'ETEYN VE ELF"
  page  383: "VEKĀYİ-İ SENE SEB‘ VE Mİ'ETEYN VE ELİF"
  page  434: 'VEKĀYİʻ-İ SENE SEMÂN VE MİʼETEYN VE ELF'
  page  518: 'VEKĀYİ‘-İ SENE TİS‘A VE MİʼETEYN VE ELF'


```python
Found 5 year marker(s), expected 6

  page  242: "VEKĀYİʻ-İ SENE HAMSÜ VE MİʼETEYN VE ELF MİN HİCRETİ MEN LEHU'L-ʻİZZÜ VE'Ş-ŞEREF"
  page  335: "VEKÂYİ-İ SENE SİTTE VE Mİ'ETEYN VE ELF"
  page  383: "VEKĀYİ-İ SENE SEB‘ VE Mİ'ETEYN VE ELİF"
  page  434: 'VEKĀYİʻ-İ SENE SEMÂN VE MİʼETEYN VE ELF'
  page  518: 'VEKĀYİ‘-İ SENE TİS‘A VE MİʼETEYN VE ELF'
```
The year that is missing is on page 192, which was one of the few-shot examples so it makes total sense that it would not have a year marker.

### Tangent: Improving the year regex

I got only one year marker in my initial attempt with VEKAYI_PATTERN = re.compile(r"VEK[AÂ]Y[İI]", re.IGNORECASE)

❌ Found 1 year marker(s), expected 6

  page  335: "VEKÂYİ-İ SENE SİTTE VE Mİ'ETEYN VE ELF"

Let's see if the issue is with my regex or with the data itself


In [ ]:
# get everything with only a subheading (body is null/empty), regardless of whether it matches the VEKĀYİ pattern

all_subheading_only = []
for jf in json_files:
    with open(jf, "r", encoding="utf-8") as f:
        wrapper = json.load(f)

    raw_response = wrapper.get("raw_response")
    if raw_response is None:
        continue

    cleaned = raw_response.strip()
    if cleaned.startswith("```"):
        lines = cleaned.splitlines()
        if lines and lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip().startswith("```"):
            lines = lines[:-1]
        cleaned = "\n".join(lines).strip()

    try:
        parsed = json.loads(cleaned)
    except json.JSONDecodeError:
        continue

    for ev in parsed.get("events", []):
        if not isinstance(ev, dict):
            continue
        sub = ev.get("subheading") or ""
        body = ev.get("body")
        body_is_empty = body is None or body == "null" or body == ""

        if body_is_empty:
            all_subheading_only.append({
                "file": jf.name,
                "page": parsed.get("page"),
                "subheading": sub
            })

print(f"\nFound {len(all_subheading_only)} events with only a subheading (body is null/empty):")
for ev in all_subheading_only:
    print(f"  - {ev['file']} (Page {ev['page']}): {ev['subheading']}")


Found 17 events with only a subheading (body is null/empty):
  - page_242.json (Page 242): VEKĀYİʻ-İ SENE HAMSÜ VE MİʼETEYN VE ELF MİN HİCRETİ MEN LEHU'L-ʻİZZÜ VE'Ş-ŞEREF
  - page_322.json (Page 322): Mâdde-i sâniye
  - page_327.json (Page 327): İcmâl-i musâlaha-i Nemçe
  - page_327.json (Page 327): Altıncı madde
  - page_328.json (Page 328): Onikinci mâdde
  - page_335.json (Page 335): VEKÂYİ-İ SENE SİTTE VE Mİ'ETEYN VE ELF
  - page_356.json (Page 356): Hubûbât Nâzırı olan Monlâcık-zâde ʻAtâ'ullah Efendi bi-hasebi't-tarîk Mekke-i mükerreme Kadısı olup, hidmeti hâlî kalmağla Kuds-i şerîf'den munfasıl İsmaʻîl Paşa-zâde Mehmed Bey hubûbâta Nâzır-ı müstakıll oldu.
  - page_376.json (Page 376): Vukūʻ bulan ahvâl netîce-i maslahata dâll ve belki gāyet-i emre hüsn-i matla u berâʻatü'l-istihlâl olduğu müsellem olduğundan, ibtidâ-yı emrde Mora Vâlîsi Vezîr Silahdâr Mustafa Paşa ile muhabere ve berren Donanma-yı hümâyûn maʻiyyetine Mora'dan 'asker irsâl olunmasının lüzûmu bi'l-mürâsele müzâker

Yes, so a part of the issue was me not accounting for Ā in VEKĀYİ! I edited that above.

## How many subheading_only events do we have that are not years?

In [16]:
def get_only_subheading_events(json_files):
    """
    Find events with only a subheadings
    remove those that are matching the VEKĀYİ pattern
    """
    VEKAYI_PATTERN = re.compile(r"VEK[AÂĀ]Y[İI]", re.IGNORECASE)
    
    found_markers = []

    for jf in json_files:
        with open(jf, "r", encoding="utf-8") as f:
            wrapper = json.load(f)

        raw_response = wrapper.get("raw_response")
        if raw_response is None:
            continue

        cleaned = raw_response.strip()
        if cleaned.startswith("```"):
            lines = cleaned.splitlines()
            if lines and lines[0].startswith("```"):
                lines = lines[1:]
            if lines and lines[-1].strip().startswith("```"):
                lines = lines[:-1]
            cleaned = "\n".join(lines).strip()

        try:
            parsed = json.loads(cleaned)
        except json.JSONDecodeError:
            continue

        for ev in parsed.get("events", []):
            if not isinstance(ev, dict):
                continue
            sub = ev.get("subheading") or ""
            body = ev.get("body")
            body_is_empty = body is None or body == "null" or body == ""

            if body_is_empty and not VEKAYI_PATTERN.search(sub):
                found_markers.append({
                    "file": jf.name,
                    "page": parsed.get("page"),
                    "subheading": sub
                })

    # ── Report ────────────────────────────────────────────────

    print(f"\nFound {len(found_markers)} events with only a subheading (body is null/empty):")

    for m in found_markers:
        print(f"  page {m['page']:>4}: {m['subheading']!r}")

    return found_markers

In [17]:
json_files = sorted(Path("gemini_output").glob("*.json"))
subheading_only = get_only_subheading_events(json_files)


Found 12 events with only a subheading (body is null/empty):
  page  322: 'Mâdde-i sâniye'
  page  327: 'İcmâl-i musâlaha-i Nemçe'
  page  327: 'Altıncı madde'
  page  328: 'Onikinci mâdde'
  page  356: "Hubûbât Nâzırı olan Monlâcık-zâde ʻAtâ'ullah Efendi bi-hasebi't-tarîk Mekke-i mükerreme Kadısı olup, hidmeti hâlî kalmağla Kuds-i şerîf'den munfasıl İsmaʻîl Paşa-zâde Mehmed Bey hubûbâta Nâzır-ı müstakıll oldu."
  page  376: "Vukūʻ bulan ahvâl netîce-i maslahata dâll ve belki gāyet-i emre hüsn-i matla u berâʻatü'l-istihlâl olduğu müsellem olduğundan, ibtidâ-yı emrde Mora Vâlîsi Vezîr Silahdâr Mustafa Paşa ile muhabere ve berren Donanma-yı hümâyûn maʻiyyetine Mora'dan 'asker irsâl olunmasının lüzûmu bi'l-mürâsele müzâkere olundukdan sonra, i'ânet-i Mürselü'r-riyâh ile Manya tarafına doğru bâd-bân-güşâ oldular. Portokava Limânı'na tekarrüb olundukda, a‘dânın ‘asker-i nuhûset-eseriyle memlû altı kıtʻa"
  page  398: "İspanya Elçisi bu esnâda devleti tarafından matlûb olduğunu mübeyyen ibr

12 mistakes! We will investigate these further below.

## How many pages have maybe too many events?

Event counts can also highlight issues with extraction. I know from having read these works that each page should have about 4 events maximum. Pages with more than 5 events need to be investigated more closely.

One major exception to this rule would be if the page contains a year change. In this case, we asked the model to treat the year change as an event with a subheading but without a body. Hence we are already printing out the cases where there is an event that looks like a year.

In [11]:
def check_high_event_counts(json_files):
    """Flag files with 5+ events and identify year markings."""
    high_event_counts = []
    output_dir = Path("gemini_output")
    
    for jf in json_files:
        with open(jf, "r", encoding="utf-8") as f:
            wrapper = json.load(f)
        
        raw_response = wrapper.get("raw_response")
        if raw_response is None:
            continue
        
        # Strip fences
        cleaned = raw_response.strip()
        if cleaned.startswith("```"):
            lines = cleaned.splitlines()
            if lines and lines[0].startswith("```"):
                lines = lines[1:]
            if lines and lines[-1].strip().startswith("```"):
                lines = lines[:-1]
            cleaned = "\n".join(lines).strip()
        
        try:
            parsed = json.loads(cleaned)
        except json.JSONDecodeError:
            continue
        
        events = parsed.get("events", [])
        
        if len(events) >= 5:
            # Check if any event is a year marking (subheading exists, body is null)
            year_markings = [
                event.get("subheading")
                for event in events
                if isinstance(event, dict) and event.get("body") is None and event.get("subheading")
            ]
            
            high_event_counts.append({
                "file": jf.name,
                "count": len(events),
                "year_markings": year_markings
            })
    
    return high_event_counts

In [12]:
json_files = sorted(output_dir.glob("page_*.json"))
high_event_counts = check_high_event_counts(json_files)

if high_event_counts:
    print(f"\n⚠️ Files with 5+ events ({len(high_event_counts)}):")
    for item in high_event_counts:
        if item['year_markings']:
            print(f"  - {item['file']} ({item['count']} events, includes year marking: {item['year_markings']})")
        else:
            print(f"  - {item['file']} ({item['count']} events)")
else:
    print("\n✅ No files with 5+ events")


⚠️ Files with 5+ events (4):
  - page_278.json (6 events)
  - page_327.json (8 events, includes year marking: ['İcmâl-i musâlaha-i Nemçe', 'Altıncı madde'])
  - page_328.json (7 events, includes year marking: ['Onikinci mâdde'])
  - page_383.json (5 events, includes year marking: ["VEKĀYİ-İ SENE SEB‘ VE Mİ'ETEYN VE ELİF"])



⚠️ Files with 5+ events (4):
  - page_278.json (6 events)
  - page_327.json (8 events, includes year marking: ['İcmâl-i musâlaha-i Nemçe', 'Altıncı madde'])
  - page_328.json (7 events, includes year marking: ['Onikinci mâdde'])
  - page_383.json (5 events, includes year marking: ["VEKĀYİ-İ SENE SEB‘ VE Mİ'ETEYN VE ELİF"])

Let's look at all of these pages and see what the issue is.
- Page 278 is has some issues which we will address below
- Page 327 is has some issues which we will address below
- Page 328 has the same issues as 327
- Page 383 is a year page. It has 4 events, plus the year change.

In [31]:
def inspect_high_event_pages(filenames):
    """
    For files with high event counts, show each event's subheading and body preview.
    """
    output_dir = Path("gemini_output")
    
    for filename in filenames:
        print(f"\n{'='*60}")
        print(f"Inspecting: {filename}")
        print('='*60)
        
        with open(output_dir / filename, "r", encoding="utf-8") as f:
            wrapper = json.load(f)
        
        raw = wrapper.get("raw_response")
        if raw is None:
            print("  ⚠️  raw_response is None")
            continue
        
        cleaned = raw.strip()
        if cleaned.startswith("```"):
            lines = cleaned.splitlines()
            if lines and lines[0].startswith("```"):
                lines = lines[1:]
            if lines and lines[-1].strip().startswith("```"):
                lines = lines[:-1]
            cleaned = "\n".join(lines).strip()
        
        try:
            parsed = json.loads(cleaned)
        except json.JSONDecodeError as e:
            print(f"  ❌ JSON parse error: {e}")
            continue
        
        events = parsed.get("events", [])
        print(f"  Total events: {len(events)}\n")
        
        for idx, ev in enumerate(events, start=1):
            if not isinstance(ev, dict):
                print(f"  Event {idx}: ❌ Not a dict\n")
                continue
            
            subheading = ev.get("subheading")
            body = ev.get("body")
            
            print(f"  Event {idx}:")
            print(f"    subheading: {subheading!r}")
            
            # Show body preview (first 150 chars) or null if body is None
            if body is None:
                body_preview = "null"
            else:
                body_preview = body[:150] + "..." if len(body) > 150 else body
            print(f"    body: {body_preview!r}\n")

In [33]:
inspect_high_event_pages(["page_278.json", "page_327.json", "page_328.json"])


Inspecting: page_278.json
  Total events: 6

  Event 1:
    subheading: None
    body: "O şey ki, davet-i miʻrâc-i kurb-ı hazret edüp,\nGehî Burak'a gehî Refref'e süvâr etdi."

  Event 2:
    subheading: "Hudavendigâr-ı esbak dâme fî rahmeti'l-Hakk [81a] hazretlerinin cülûsları târîhidir:"
    body: 'Dedi Tevfik bekā-hâhî ona târîh-i tâm,\nDevlet ü mecd ile Sultân Mustafa kıldı cülûs.'

  Event 3:
    subheading: "Sâhib-kırân-ı zemân dâme fî-hifzi'l-Müsteʻân hazretlerinin velâdet-i pür-meymenetleri târîhidir:"
    body: 'Bendesi Tevfik târîhin duʻâ edüp dedi,\nGeldi kevne devlet ile pîr ola Sultân Selîm.'

  Event 4:
    subheading: "Şâh Sultân-ı ‘aliyyetü'ş-şân hazretlerinin velâdet târîhidir:"
    body: "Dedi Tevfikâ beşâret birle târîh-i temâm,\nŞâh Sultân tâliʻ oldu müjde nesl-i Şâh'dan."

  Event 5:
    subheading: 'Bî-nukat-ı gazeliyyâttındandır:'
    body: "Der ki, darü'l-hümûm-ı dil o dem maʻmûr olur,\nGer esâs-ı 'ahd-ı dildârın olur muhkem sana,\nLûh-ı dil-âlûde-i gerd-i hümû

`Page 327`

```python

============================================================
Inspecting: page_327.json
============================================================
  Total events: 8

  Event 1:
    subheading: None
    body: "Seyyid ʻAbdullah Efendi'ye on kîse ve sânî vü sâlise beşer kîse senevî îrâd ihsân olunup, bu vesîle ile mahsûdü'l-akrân ve mahfûf-ı ʻavârif-i bî-hadd ..."

  Event 2:
    subheading: 'İcmâl-i musâlaha-i Nemçe'
    body: 'null'

  Event 3:
    subheading: 'Birinci mâdde'
    body: "Devleteyn beyninde vâkiʻ olan muhârebe refʻ ve iki tarafın mücrimleri ʻafv ve tarafeyne bi'l-irâde tâbiʻ olanlara sûret-i menʻ gösterilmeyüp, havf-ı c..."

  Event 4:
    subheading: 'İkinci mâdde'
    body: "Elli iki senesinden işbu muhârebe evveline gelince beyne'd-devleteyn cârî olan ʻahid-nâmeler merʻî ve istatüsko isterkat taʻbîrinin hükmü işbu musâlah..."

  Event 5:
    subheading: 'Üçüncü mâdde'
    body: "Garb Ocağları'nın Nemçe sefâyinine taʻarruzların vukūʻunda hasâretleri Devlet-i ʻaliyye tarafından tazmîn ve bi'l-cümle bihâr u enhârda Nemçelü'nün se..."

  Event 6:
    subheading: 'Dördüncü mâdde'
    body: "İşbu muhârebede Nemçelü tarafından zabt olununan kılâʻ vü erâzî istatüsko isterkat şartıyla Devlet-i ʻaliyye'ye redd olunup, hudûd-ı kadîme ibkā ve âl..."

  Event 7:
    subheading: 'Beşinci mâdde'
    body: "Hotin Kalʻası ve kazâsında vâkiʻ reʻâyâ-yı Rûsiyyelü ile sulh tanzîm olununcaya dek ber-veche-i emânet Nemçelü tarafında kalup, Rûsiyyelü'ye vechen mi..."

  Event 8:
    subheading: 'Altıncı madde'
    body: 'null'
```
![img](page_327.png)

This page contains the clauses of a treaty signed with the Austrians. As we can see in this example, there are a few issues. 

    Firstly, the second event ('İcmâl-i musâlaha-i Nemçe') looks structurally like a year but it is actually the subheading of the whole event, which consists of the treaty clauses. 

    Second, the 8th event ('Altıncı madde') refers to clause 6 and because the page ended before the clause itself, it ended up being structured like a year. 

There are two solutions to this. The first one is that the model needs to explicitly see that treaty clauses and these kinds of lists should not be split into individual events. I have seen this happen in another work as well and added a few-shot example that specifically shows what I want in this case.

We also saw here that there are outlier cases where an event could look like a year but it is not. We must account for this in our post-processing and make sure to have a secondary check for the year headings, like checking if the word 'sene' or 'vekayi' or some other version of these exists in the text.

Also, we should make sure to handle the events with body set to null in the same way we handle the ones where body does not exist.

But how do we fix these 2 pages? We can redo the OCR for these pages but we don't have to. We will extract these events in the post and merge them manually. To help us keep track of this, we will create another check: does the event have 'madde' or another word that the model sometimes mistakes for a subheading in its subheading?


## Checking subheadings for some words the model commonly mistakes for subheading

ie: madde, nazm, mısra

can you write the code so that we loop over all the subheadings and see where these words appear? madde oculd also be spelled as mâdde so keep that in mind for the other words too


In [ ]:
def check_subheading_keywords(json_files):
    """Flag events whose subheadings contain commonly mistaken words: madde/mâdde, nazm/nâzm, mısra/mîsra."""
    suspect_events = []
    output_dir = Path("gemini_output")

    SUSPECT_PATTERNS = [
        re.compile(r'm[aâ]dde', re.IGNORECASE), # madde
        re.compile(r'n[aâ]zm',  re.IGNORECASE), # nazm
        re.compile(r'n[aâ]z[iıî]m',  re.IGNORECASE), # nazım
        re.compile(r'm[iîı]sra', re.IGNORECASE), # mısra
        re.compile(r':', re.IGNORECASE), # : usually smt like tarihidir: or nazm: which is not a subheading but a continuation of the body
    ]

    for jf in json_files:
        with open(jf, "r", encoding="utf-8") as f:
            wrapper = json.load(f)

        raw_response = wrapper.get("raw_response")
        if raw_response is None:
            continue

        # Strip fences
        cleaned = raw_response.strip()
        if cleaned.startswith("```"):
            lines = cleaned.splitlines()
            if lines and lines[0].startswith("```"):
                lines = lines[1:]
            if lines and lines[-1].strip().startswith("```"):
                lines = lines[:-1]
            cleaned = "\n".join(lines).strip()

        try:
            parsed = json.loads(cleaned)
        except json.JSONDecodeError:
            continue

        events = parsed.get("events", [])

        hits = [
            {"event_index": idx, "subheading": ev.get("subheading")}
            for idx, ev in enumerate(events, start=1)
            if isinstance(ev, dict)
            and ev.get("subheading")
            and any(pat.search(ev["subheading"]) for pat in SUSPECT_PATTERNS)
        ]

        if hits:
            suspect_events.append({
                "file": jf.name,
                "hits": hits
            })

    return suspect_events

In [22]:
json_files = sorted(Path("gemini_output").glob("*.json"))
results = check_subheading_keywords(json_files)

for r in results:
    print(f"\n{r['file']} — {len(r['hits'])} hit(s)")
    for h in r['hits']:
        print(f"  Event {h['event_index']}: {h['subheading']!r}")


page_144.json — 1 hit(s)
  Event 2: 'Hayrî Efendi merhûmun zebân-âver-i beyân olduğu târîhdir: Nazım:'

page_145.json — 2 hit(s)
  Event 2: "Nâşid Bey'in inşâd eylediği târîhdir, [Nazm]ʻ:"
  Event 3: "Rebîʻ-i mevsim-i ʻirfân olan behâr-ı Şîrâzî'nin nâtıka-güzâr-ı belâgat olduğu târîh-i sihr-i helâl ve târîhi ta‘miyesinden ve ta‘miyesi târîhinden a‘lâ olduğu zâhir-i hâl olmağla, sebt-i ceride-i tezkâr ve kayd-ı mecelle-i âsâr kılındı. Nazım:"

page_278.json — 4 hit(s)
  Event 2: "Hudavendigâr-ı esbak dâme fî rahmeti'l-Hakk [81a] hazretlerinin cülûsları târîhidir:"
  Event 3: "Sâhib-kırân-ı zemân dâme fî-hifzi'l-Müsteʻân hazretlerinin velâdet-i pür-meymenetleri târîhidir:"
  Event 4: "Şâh Sultân-ı ‘aliyyetü'ş-şân hazretlerinin velâdet târîhidir:"
  Event 5: 'Bî-nukat-ı gazeliyyâttındandır:'

page_322.json — 2 hit(s)
  Event 3: 'Mâdde-i ûlâ'
  Event 4: 'Mâdde-i sâniye'

page_323.json — 1 hit(s)
  Event 2: 'Mâdde-i sâlise'

page_327.json — 6 hit(s)
  Event 3: 'Birinci mâdde'
  Event 4: 'İ


page_145.json — 1 hit(s)
  Event 2: "Nâşid Bey'in inşâd eylediği târîhdir, [Nazm]ʻ:"

page_322.json — 2 hit(s)
  Event 3: 'Mâdde-i ûlâ'
  Event 4: 'Mâdde-i sâniye'

page_323.json — 1 hit(s)
  Event 2: 'Mâdde-i sâlise'

page_327.json — 6 hit(s)
  Event 3: 'Birinci mâdde'
  Event 4: 'İkinci mâdde'
  Event 5: 'Üçüncü mâdde'
  Event 6: 'Dördüncü mâdde'
  Event 7: 'Beşinci mâdde'
  Event 8: 'Altıncı madde'

page_328.json — 6 hit(s)
  Event 2: 'Yedinci mâdde'
  Event 3: 'Sekizinci mâdde'
  Event 4: 'Dokuzuncu mâdde'
  Event 5: 'Onuncu mâdde'
  Event 6: 'Onbirinci mâdde'
  Event 7: 'Onikinci mâdde'

page_329.json — 2 hit(s)
  Event 2: 'Onüçüncü mâdde'
  Event 3: 'Ondördüncü mâdde'

page_399.json — 1 hit(s)
  Event 2: "Rumeli ve Anadolu'da vâki‘ memâlik-i Pâdişâhî'de 'adl ü dâdın intişârı ve fukarâ vü zuʻafânın ıslâh-[152a]kârı ve sâye-i şefekat vâyelerinde asayiş ü râhatla istikrârı matlûb bulup, zuhûr-ı mezâlim ise fî-ekseri'l-evkāt hükkâm u a'yân nâmıyla memleketlere müstevlî olan li’âm-ı enâmdan neş'et eyleyüp, şöyle ki, bir senelik mesârif nâmıyla ehâlî-yi kazâya tevzî‘ eyledikleri mebâliğin ‘öşrüyle umûr-ı kazâ idâre olunup, fazlası miyânelerinde münkasim ve bu sebeble reʻâyâda tâb u tüvân kalmayup, tekâlîf-i şer‘iyyeyi bile edâda tâkatleri mün‘adim olup, giderek bu zulmün istimrârı vîrân-ı memâlike sebeb ve belki mûceb-i şûr u şegab olacağı rûz u şeb hâst-gâr-ı rızâ-yı Rabb olan Şehinşâh-ı vâlâ-neseb hazretlerinin meczûmu olmağla, fî-mâ-ba'd altı mâhda bir kerre her kazânın sicillât-ı mesafiri Âsitâne-i sa‘âdet'e getirdilüp, me'mûrlar huzûrunda ta'dîl ve sıhhate makrûn olan mesârif isbât ve mevzû'ât-ı hükkâmdan olan emvâl-i zâyide hatt u tenzîl olunmak zımnında, Memâlik-i mahrûse'ye neşr-i evâmir ve ta‘yîn-i mübâşir kılınup, bu vesîle-i cemîle ile eyâdî-yi zulme kûtâh ve me’lûf-ı ekl ü bel‘ olan rû-siyâhlar mübtelâ-yı nâliş-i cângâh oldular. Zikrolunan defâtirin tenkīhi ve ıskāt-ı zevâyid ve hazf-ı avâyid ile tashîhi ahvâl-i memâlike muttali‘ bir zâta tahsîs olunmak lâzım geldiğine binâ'en, Anadolu Defterleri, Şıkk-ı Evvel-i sâbık Mehmed Şerîf Efendi'ye ve Rumeli Defterleri, Yenişehirli Mustafa Bey'e tefvîz olundu. Nazm: Geldi mîzân cemâl-i âraya ey Yusuf Hasan! Korkarım çok keşenk ipliği bâzâra çıkar."

Most of these examples showed the same issue as we had with page 327: clauses are mistaken for subheadings. We will flag them and fix them in the post processing.

## Are there any subheadings longer than the body of text associated with them?

This is an issue related to lack of body in the schema or body being marked as null even if the example is not a year. In most cases it is a mistake and the subheading should be merged with the body of the previous event.

But there can be some edge cases where an event is at the very bottom of the page and it ends up being longer due to coincidence. 

In [45]:
def check_subheading_longer_than_body(json_files):
    """Flag events where the subheading is longer than its associated body text."""
    long_subheadings = []

    for jf in json_files:
        with open(jf, "r", encoding="utf-8") as f:
            wrapper = json.load(f)

        raw_response = wrapper.get("raw_response")
        if raw_response is None:
            continue

        # Strip fences
        cleaned = raw_response.strip()
        if cleaned.startswith("```"):
            lines = cleaned.splitlines()
            if lines and lines[0].startswith("```"):
                lines = lines[1:]
            if lines and lines[-1].strip().startswith("```"):
                lines = lines[:-1]
            cleaned = "\n".join(lines).strip()

        try:
            parsed = json.loads(cleaned)
        except json.JSONDecodeError:
            continue

        events = parsed.get("events", [])

        hits = [
            {
                "event_index": idx,
                "subheading": ev.get("subheading"),
                "subheading_len": len(ev.get("subheading") or ""),
                "body": ev.get("body"),
                "body_len": len(ev.get("body") or ""),
            }
            for idx, ev in enumerate(events, start=1)
            if isinstance(ev, dict)
            and ev.get("subheading")
            and len(ev.get("subheading") or "") > len(ev.get("body") or "")
        ]


        if hits:
            long_subheadings.append({
                "file": jf.name,
                "hits": hits
            })

    return long_subheadings

In [46]:
json_files = sorted(Path("gemini_output").glob("*.json"))
results = check_subheading_longer_than_body(json_files)

for r in results:
    print(f"\n{r['file']} — {len(r['hits'])} hit(s)")
    for h in r['hits']:
        print(f"  Event {h['event_index']}: subheading({h['subheading_len']} chars) > body({h['body_len']} chars)")
        print(f"    subheading: {h['subheading']!r}")


page_242.json — 1 hit(s)
  Event 2: subheading(79 chars) > body(0 chars)
    subheading: "VEKĀYİʻ-İ SENE HAMSÜ VE MİʼETEYN VE ELF MİN HİCRETİ MEN LEHU'L-ʻİZZÜ VE'Ş-ŞEREF"

page_278.json — 1 hit(s)
  Event 3: subheading(96 chars) > body(83 chars)
    subheading: "Sâhib-kırân-ı zemân dâme fî-hifzi'l-Müsteʻân hazretlerinin velâdet-i pür-meymenetleri târîhidir:"

page_322.json — 1 hit(s)
  Event 4: subheading(14 chars) > body(0 chars)
    subheading: 'Mâdde-i sâniye'

page_327.json — 2 hit(s)
  Event 2: subheading(24 chars) > body(0 chars)
    subheading: 'İcmâl-i musâlaha-i Nemçe'
  Event 8: subheading(13 chars) > body(0 chars)
    subheading: 'Altıncı madde'

page_328.json — 1 hit(s)
  Event 7: subheading(14 chars) > body(0 chars)
    subheading: 'Onikinci mâdde'

page_335.json — 1 hit(s)
  Event 2: subheading(38 chars) > body(0 chars)
    subheading: "VEKÂYİ-İ SENE SİTTE VE Mİ'ETEYN VE ELF"

page_356.json — 1 hit(s)
  Event 2: subheading(210 chars) > body(0 chars)
    subheading: "Hubû

We learnt another flag here. If the body is significantly shorter, we can actually just merge it with the event above in the post-processing

## What is a good heuristic for saying that a subheading is too long for the body?

Here we will combine all of our previous learnings. Let's get some distributions about the subheading lengths first.

- number of subheadings that are not null
- number of subheadings that have null in body
  - of this number, how many are year and how many are not
  - year ones are excluded from the rest of the calculations
  - non year subheadings with no body will be edited in post-processing -> this is one of the problems we are addressing
- number of subheadings that are longer than their body text -> this is one of the problems we are addressing
- number of subheadings that have body and are shorter than their body text -> this is our normal

Then we will get the average length of subheading based on our normal and we will use that to see if the problem subheadings are too long or short from a statistical point

In [ ]:
def check_subheading_distributions(json_files):
    """
    Get distributions of subheading lengths to inform
    reasonable thresholds for post-processing fixes.
    """
    all_subheadings = []  # all non-null subheadings with their context

    VEKAYI_PATTERN = re.compile(r"VEK[AÂĀ]Y[İI]", re.IGNORECASE)

    for jf in json_files:
        with open(jf, "r", encoding="utf-8") as f:
            wrapper = json.load(f)

        raw_response = wrapper.get("raw_response")
        if raw_response is None:
            continue

        cleaned = raw_response.strip()
        if cleaned.startswith("```"):
            lines = cleaned.splitlines()
            if lines and lines[0].startswith("```"):
                lines = lines[1:]
            if lines and lines[-1].strip().startswith("```"):
                lines = lines[:-1]
            cleaned = "\n".join(lines).strip()

        try:
            parsed = json.loads(cleaned)
        except json.JSONDecodeError:
            continue

        for ev in parsed.get("events", []):
            if not isinstance(ev, dict):
                continue
            sub = ev.get("subheading")
            if not sub:
                continue
            body = ev.get("body")
            body_is_empty = body is None or body == "null" or body == ""
            body_len = len(body or "")
            sub_len = len(sub)
            is_year = bool(VEKAYI_PATTERN.search(sub)) and body_is_empty

            all_subheadings.append({
                "file":         jf.name,
                "subheading":   sub,
                "sub_len":      sub_len,
                "body_len":     body_len,
                "body_is_empty": body_is_empty,
                "is_year":      is_year,
            })

    # ── Buckets ───────────────────────────────────────────────
    year_markers   = [s for s in all_subheadings if s["is_year"]]
    null_body      = [s for s in all_subheadings if s["body_is_empty"] and not s["is_year"]]
    longer_than_body = [
        s for s in all_subheadings
        if not s["body_is_empty"] and not s["is_year"]
        and s["sub_len"] > s["body_len"]
    ]
    normal         = [
        s for s in all_subheadings
        if not s["body_is_empty"] and not s["is_year"]
        and s["sub_len"] <= s["body_len"]
    ]

    # ── Stats on normal subheadings ───────────────────────────
    normal_sub_lens  = [s["sub_len"]  for s in normal]
    normal_body_lens = [s["body_len"] for s in normal]

    def stats(vals):
        if not vals:
            return {}
        return {
            "count":  len(vals),
            "mean":   round(mean(vals), 1),
            "median": round(median(vals), 1),
            "min":    min(vals),
            "max":    max(vals),
        }

    # ── Report ────────────────────────────────────────────────
    print(f"Total non-null subheadings: {len(all_subheadings)}\n")

    print(f"  Year markers (null body + VEKĀYİ):      {len(year_markers)}")
    print(f"  Null body, non-year (⚠️  problem type 1): {len(null_body)}")
    print(f"  Subheading longer than body (⚠️  problem type 2): {len(longer_than_body)}")
    print(f"  Normal (subheading ≤ body):              {len(normal)}\n")

    print(f"{'─'*60}")
    print(f"Normal subheadings — length stats:")
    s = stats(normal_sub_lens)
    print(f"  subheading chars — mean: {s['mean']}, median: {s['median']}, min: {s['min']}, max: {s['max']}")
    s = stats(normal_body_lens)
    print(f"  body chars       — mean: {s['mean']}, median: {s['median']}, min: {s['min']}, max: {s['max']}")

    print(f"\n{'─'*60}")
    print(f"Problem type 1 — null body, non-year subheadings:")
    for s in null_body:
        flag = "  🔴 VERY LONG" if s["sub_len"] > 3 * median(normal_sub_lens) else ""
        print(f"  {s['file']} | sub_len: {s['sub_len']}{flag}")
        print(f"    {s['subheading'][:120]!r}")

    print(f"\n{'─'*60}")
    print(f"Problem type 2 — subheading longer than body:")
    for s in longer_than_body:
        flag = "  🔴 VERY LONG" if s["sub_len"] > median(normal_sub_lens) else ""
        print(f"  {s['file']} | sub_len: {s['sub_len']} > body_len: {s['body_len']}{flag}")
        print(f"    {s['subheading'][:120]!r}")

    return {
        "all":             all_subheadings,
        "year_markers":    year_markers,
        "null_body":       null_body,
        "longer_than_body": longer_than_body,
        "normal":          normal,
        "normal_sub_stats": stats(normal_sub_lens),
    }

In [26]:
from statistics import mean, median

json_files = sorted(Path("gemini_output").glob("*.json"))
dist = check_subheading_distributions(json_files)

Total non-null subheadings: 434

  Year markers (null body + VEKĀYİ):      5
  Null body, non-year (⚠️  problem type 1): 12
  Subheading longer than body (⚠️  problem type 2): 3
  Normal (subheading ≤ body):              414

────────────────────────────────────────────────────────────
Normal subheadings — length stats:
  subheading chars — mean: 40.9, median: 36.0, min: 6, max: 362
  body chars       — mean: 796.1, median: 678.0, min: 66, max: 2425

────────────────────────────────────────────────────────────
Problem type 1 — null body, non-year subheadings:
  page_322.json | sub_len: 14
    'Mâdde-i sâniye'
  page_327.json | sub_len: 24
    'İcmâl-i musâlaha-i Nemçe'
  page_327.json | sub_len: 13
    'Altıncı madde'
  page_328.json | sub_len: 14
    'Onikinci mâdde'
  page_356.json | sub_len: 210  🔴 VERY LONG
    "Hubûbât Nâzırı olan Monlâcık-zâde ʻAtâ'ullah Efendi bi-hasebi't-tarîk Mekke-i mükerreme Kadısı olup, hidmeti hâlî kalmağ"
  page_376.json | sub_len: 481  🔴 VERY LONG
    "V

## Can subheadings be too short?

In [27]:
def parse_raw_response(raw):
    """Strip markdown fences and parse JSON. Returns (parsed_dict, error_str)."""
    if raw is None:
        return None, "raw_response is None"

    cleaned = raw.strip()
    if cleaned.startswith("```"):
        lines = cleaned.splitlines()
        if lines and lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip().startswith("```"):
            lines = lines[:-1]
        cleaned = "\n".join(lines).strip()

    try:
        return json.loads(cleaned), None
    except json.JSONDecodeError as e:
        return None, str(e)

In [28]:
def check_short_subheadings(json_files):
    """
    Explore suspiciously short subheadings using statistical thresholds
    derived from the normal subheading distribution.
    """
    normal_lens = []
    all_subheadings = []

    VEKAYI_PATTERN = re.compile(r"VEK[AÂĀ]Y[İI]", re.IGNORECASE)

    for jf in json_files:
        with open(jf, "r", encoding="utf-8") as f:
            wrapper = json.load(f)

        parsed, err = parse_raw_response(wrapper.get("raw_response"))
        if err:
            continue

        for ev in parsed.get("events", []):
            if not isinstance(ev, dict):
                continue
            sub  = ev.get("subheading")
            if not sub:
                continue
            body = ev.get("body")
            body_is_empty = body is None or body == "null" or body == ""
            is_year = bool(VEKAYI_PATTERN.search(sub)) and body_is_empty

            if is_year:
                continue

            sub_len  = len(sub)
            body_len = len(body or "")

            all_subheadings.append({
                "file":    jf.name,
                "sub_len": sub_len,
                "body_len": body_len,
                "subheading": sub,
                "body_is_empty": body_is_empty,
            })

            # Only normal events contribute to the distribution
            if not body_is_empty and sub_len <= body_len:
                normal_lens.append(sub_len)

    if not normal_lens:
        print("Not enough normal subheadings to compute thresholds.")
        return

    # ── Thresholds ────────────────────────────────────────────
    sorted_lens = sorted(normal_lens)
    mid = len(sorted_lens) // 2
    q1  = median(sorted_lens[:mid])
    q3  = median(sorted_lens[mid:] if len(sorted_lens) % 2 == 0 else sorted_lens[mid+1:])
    iqr = q3 - q1

    thresh_iqr    = q1 - 1.5 * iqr          # classic outlier floor
    thresh_q1     = q1                        # bottom quartile
    thresh_median = median(normal_lens) * 0.25  # 25% of median

    print(f"Normal subheading distribution (n={len(normal_lens)}):")
    print(f"  Q1: {q1:.0f} chars,  median: {median(normal_lens):.0f} chars,  Q3: {q3:.0f} chars,  IQR: {iqr:.0f}")
    print(f"\nThresholds for 'short':")
    print(f"  T1 — IQR floor  (Q1 - 1.5×IQR): {thresh_iqr:.0f} chars")
    print(f"  T2 — Q1:                          {thresh_q1:.0f} chars")
    print(f"  T3 — 25% of median:               {thresh_median:.0f} chars")

    for label, thresh in [("T1 (IQR floor)", thresh_iqr), ("T2 (Q1)", thresh_q1), ("T3 (25% of median)", thresh_median)]:
        flagged = [s for s in all_subheadings if s["sub_len"] < thresh]
        print(f"\n{'─'*60}")
        print(f"{label} — subheadings under {thresh:.0f} chars ({len(flagged)} hits):")
        for s in flagged:
            body_note = "null body" if s["body_is_empty"] else f"body: {s['body_len']} chars"
            print(f"  {s['file']} | sub_len: {s['sub_len']} | {body_note}")
            print(f"    {s['subheading']!r}")

In [29]:
json_files = sorted(Path("gemini_output").glob("*.json"))
dist_low = check_short_subheadings(json_files)

Normal subheading distribution (n=414):
  Q1: 21 chars,  median: 36 chars,  Q3: 51 chars,  IQR: 30

Thresholds for 'short':
  T1 — IQR floor  (Q1 - 1.5×IQR): -24 chars
  T2 — Q1:                          21 chars
  T3 — 25% of median:               9 chars

────────────────────────────────────────────────────────────
T1 (IQR floor) — subheadings under -24 chars (0 hits):

────────────────────────────────────────────────────────────
T2 (Q1) — subheadings under 21 chars (99 hits):
  page_136.json | sub_len: 10 | body: 401 chars
    'Amma baʻdü'
  page_150.json | sub_len: 17 | body: 160 chars
    'ʻAzl-i Çavuş-başı'
  page_157.json | sub_len: 6 | body: 474 chars
    'Tezyîl'
  page_167.json | sub_len: 6 | body: 227 chars
    'Tezyîl'
  page_172.json | sub_len: 20 | body: 858 chars
    'Âmeden-i Elçi-yi Leh'
  page_174.json | sub_len: 19 | body: 357 chars
    'Tebrîk-i ʻÎd-i fıtr'
  page_184.json | sub_len: 8 | body: 1163 chars
    'İstitrâd'
  page_195.json | sub_len: 6 | body: 1317 chars

Well there is not much we can do computationally with short subheadings. They are by and large real subheadings

## Output token averages

Finally, we want to use the output token information as a proxy to understand if text extraction is complete. Since each page is OCR'ed one by one and each page is the same size and thus has similar amount of text, the output sizes should also be very similar.

Some things to keep in mind:
- the last page can be shorter
- if there are sections, the last page of a given section can be shorter

In [47]:
# Analyze token usage to flag outliers
if not output_dir.exists():
    print(f"Output directory not found: {output_dir}")
else:
    json_files = sorted(output_dir.glob("page_*.json"))
    if not json_files:
        print("No JSON files found")
    else:
        output_tokens = []
        prompt_tokens = []
        total_tokens = []
        missing_usage = []
        missing_output = []
        missing_prompt = []

        for json_file in json_files:
            with open(json_file, "r", encoding="utf-8") as f:
                wrapper = json.load(f)

            usage = wrapper.get("usage")
            if not isinstance(usage, dict):
                missing_usage.append(json_file.name)
                continue

            out_tok = usage.get("output_tokens")
            prm_tok = usage.get("prompt_tokens")
            tot_tok = usage.get("total_tokens")

            if out_tok is None:
                missing_output.append(json_file.name)
            else:
                output_tokens.append((json_file.name, out_tok))

            if prm_tok is None:
                missing_prompt.append(json_file.name)
            else:
                prompt_tokens.append((json_file.name, prm_tok))
            
            if tot_tok is not None:
                total_tokens.append(tot_tok)

        def describe(nums):
            if not nums:
                return None
            return {
                "count": len(nums),
                "mean": mean(nums),
                "median": median(nums),
                "min": min(nums),
                "max": max(nums),
            }

        def iqr_bounds(nums):
            if len(nums) < 4:
                return None
            sorted_nums = sorted(nums)
            mid = len(sorted_nums) // 2
            lower = sorted_nums[:mid]
            upper = sorted_nums[mid:] if len(sorted_nums) % 2 == 0 else sorted_nums[mid + 1:]
            q1 = median(lower)
            q3 = median(upper)
            iqr = q3 - q1
            return (q1 - 1.5 * iqr, q3 + 1.5 * iqr)

        def parse_raw_response(raw_response):
            if raw_response is None:
                return None, "missing"
            cleaned = raw_response.strip()
            if cleaned.startswith("```"):
                lines = cleaned.splitlines()
                if lines and lines[0].startswith("```"):
                    lines = lines[1:]
                if lines and lines[-1].strip().startswith("```"):
                    lines = lines[:-1]
                cleaned = "\n".join(lines).strip()
            try:
                return json.loads(cleaned), None
            except json.JSONDecodeError:
                return None, "invalid"
        
        def check_for_poetry(parsed):
            """Check if parsed JSON contains poetry markers like [Beyt:], [Mısraʿ:], etc."""
            if not isinstance(parsed, dict):
                return False
            events = parsed.get("events")
            if not isinstance(events, list):
                return False
            
            # Poetry markers with optional brackets, colons, and apostrophes
            poetry_markers = ["Beyt", "Mısra", "Kıt'a", "Kıtʿa", "Rubâ'î", "Rubâʿî", "Nazm"]
            
            for event in events:
                if not isinstance(event, dict):
                    continue
                body = event.get("body")
                if isinstance(body, str):
                    if any(marker in body for marker in poetry_markers):
                        return True
            return False
        
        def check_for_multiple_ms_pages(parsed):
            """Check if parsed JSON contains multiple manuscript page markers like (3b), (4a), etc."""
            if not isinstance(parsed, dict):
                return False
            events = parsed.get("events")
            if not isinstance(events, list):
                return False
            
            import re
            # Pattern to match manuscript page markers like [3b], (4a), (123a), etc.
            ms_page_pattern = r'[\[\(][0-9]{1,3}[ab][\]\)]'
            
            all_pages = set()
            for event in events:
                if not isinstance(event, dict):
                    continue
                body = event.get("body")
                if isinstance(body, str):
                    # Find all manuscript page markers in this event's body
                    matches = re.findall(ms_page_pattern, body)
                    all_pages.update(matches)
            
            # If more than one unique manuscript page marker found, return True
            return len(all_pages) > 1

        # Output tokens summary
        out_values = [v for _, v in output_tokens]
        print("Output tokens summary:")
        stats = describe(out_values)
        if stats:
            print(stats)
        else:
            print("No output token values found")

        # Prompt tokens - just check for outliers
        if prompt_tokens:
            prm_values = [v for _, v in prompt_tokens]
            prm_bounds = iqr_bounds(prm_values)
            if prm_bounds:
                prm_low, prm_high = prm_bounds
                prm_outliers = [(name, val) for name, val in prompt_tokens if val < prm_low or val > prm_high]
                if prm_outliers:
                    print(f"⚠️ {len(prm_outliers)} prompt token outliers:")
                    for name, val in prm_outliers:
                        print(f"  - {name}: {val}")
                else:
                    print("✅ All prompt tokens within expected range")
            else:
                print("⚠️ Not enough data for prompt token outlier detection")

        if missing_usage:
            print(f"\n❌ Missing usage block in {len(missing_usage)} files:")
            for name in missing_usage:
                print(f"  - {name}")
                
        if missing_prompt:
            print(f"\n❌ Missing prompt_tokens in {len(missing_prompt)} files:")
            for name in missing_prompt:
                print(f"  - {name}")

        # Outlier detection on output tokens
        bounds = iqr_bounds(out_values)
        if bounds:
            low, high = bounds
            low_outliers = [(name, val) for name, val in output_tokens if val < low]
            high_outliers = [(name, val) for name, val in output_tokens if val > high]
            
            print(f"\n{len(low_outliers)} low outliers, {len(high_outliers)} high outliers")

            
            # Check subheading_note and multiple ms pages for high outliers
            if high_outliers:
                print("\nHigh outliers:")
                for name, val in high_outliers:
                    with open(output_dir / name, "r", encoding="utf-8") as f:
                        wrapper = json.load(f)
                    parsed, err = parse_raw_response(wrapper.get("raw_response"))
                    
                    flags = []
                    
                    if err:
                        flags.append(f"raw_response {err}")
                    else:
                        events = parsed.get("events")
                        if not isinstance(events, list):
                            flags.append("events missing or not list")
                        else:
                            # Check for subheading_note
                            has_note = any(
                                isinstance(event, dict) and event.get("subheading_note") not in (None, "")
                                for event in events
                            )
                            if has_note:
                                flags.append("SUBHEADING_NOTE")
                            
                            # Check for multiple manuscript pages
                            has_multi_ms_pages = check_for_multiple_ms_pages(parsed)
                            if has_multi_ms_pages:
                                flags.append("MULTI-MS-PAGES")
                    
                    flag_str = f" [{', '.join(flags)}]" if flags else ""
                    print(f"  - {name}: {val}{flag_str}")

                    # Print low outliers with first/last page marking, poetry check, and multi-page check
                    if low_outliers:
                        print("\nLow outliers:")
                        for name, val in low_outliers:
                            # Extract page number from filename
                            page_match = re.search(r'page_(\d+)\.json', name)
                            page_num = int(page_match.group(1)) if page_match else None
                            
                            # Check for poetry
                            with open(output_dir / name, "r", encoding="utf-8") as f:
                                wrapper = json.load(f)
                            parsed, err = parse_raw_response(wrapper.get("raw_response"))
                            has_poetry = check_for_poetry(parsed) if parsed else False
                            
                            flags = []
                            if page_num == 1 or (page_match and page_match.group(1) == "1"):
                                flags.append("FIRST PAGE")
                            if page_num == pdf_page_count:
                                flags.append("LAST PAGE")
                            if has_poetry:
                                flags.append("POETRY")
                            
                            flag_str = f" [{', '.join(flags)}]" if flags else ""
                            print(f"  - {name}: {val}{flag_str}")

Output tokens summary:
{'count': 384, 'mean': 924.3984375, 'median': 938.0, 'min': 398, 'max': 1244}
⚠️ 58 prompt token outliers:
  - page_134.json: 5310
  - page_135.json: 5310
  - page_136.json: 5310
  - page_137.json: 5310
  - page_138.json: 5310
  - page_139.json: 5310
  - page_140.json: 5310
  - page_141.json: 5310
  - page_142.json: 5310
  - page_143.json: 5310
  - page_144.json: 5310
  - page_145.json: 5310
  - page_146.json: 5310
  - page_147.json: 5310
  - page_148.json: 5310
  - page_149.json: 5310
  - page_150.json: 5310
  - page_151.json: 5378
  - page_152.json: 5378
  - page_153.json: 5378
  - page_155.json: 5378
  - page_156.json: 5378
  - page_157.json: 5378
  - page_158.json: 5378
  - page_159.json: 5378
  - page_160.json: 5378
  - page_161.json: 5378
  - page_162.json: 5378
  - page_163.json: 5378
  - page_164.json: 5378
  - page_165.json: 5378
  - page_166.json: 5378
  - page_167.json: 5378
  - page_168.json: 5378
  - page_169.json: 5378
  - page_170.json: 5378
  - pa

The issue with this work is that as I was experimenting, I changed the number of few-shot examples. Hence the check about prompt length is irrelevant. Similarly pages 154, 192, and 216 are the few-shot examples, so they are not important here either. We need to check the high and low outliers

High outliers:
  - page_418.json: 1244 [MULTI-MS-PAGES]

Low outliers:
  - page_135.json: 465
  - page_157.json: 528
  - page_166.json: 621
  - page_197.json: 626 [POETRY]
  - page_199.json: 607 [POETRY]
  - page_275.json: 620
  - page_277.json: 593 [POETRY]
  - page_278.json: 638
  - page_353.json: 485
  - page_357.json: 635
  - page_385.json: 589
  - page_453.json: 522 [POETRY]
  - page_461.json: 613 [POETRY]
  - page_480.json: 620
  - page_494.json: 594
  - page_504.json: 588
  - page_520.json: 398 [POETRY]

The idea behind marking multi manuscript pages is that if there are no footnotes, there is more space on the page and the author gets to publish more of the transliteration on a single page. One way to account for that is to see if more of the transliteration is actually on this page by seeing how many manuscript pages are marked on the PDF page. This is indeed the case on page 418

![img](page_418.png)

Marking poetry follows a similar logic. If there is poetry, it means that there is a lot more empty space around the lines, hence there is less text on a single page. This would then translate to less text in the output.

We can see this on page 461

![img](page_461.png)

Finally, what about the unmarked ones? A quick look at these pages show that they have really long footnotes. They are also only 17 pages including the ones with poetry out of 387, which is a small percentage of the whole text.